# Regresi Lag Spasial Curah Hujan vs Nino-3.4

Notebook ini menghitung regresi lag spasial untuk curah hujan DJF terhadap Nino-3.4 rata-rata berjalan 3 bulan yang terpusat, lalu menyimpan cache NetCDF agar plot dapat diubah tanpa menghitung ulang.

Yang dihitung:
- 1981-2025
- P1 = 1981-2006
- P2 = 2007-2025
- selisih P2-P1

Notebook ini memakai gaya peta yang sama seperti `plot_regression_mc.ipynb`.


## Konvensi Lag

- `lag 0` = `DJF`
- `lag -1` = `NDJ`
- `lag -2` = `OND`
- `lag -3` = `SON`
- `lag 1` = `JFM`
- `lag 2` = `FMA`
- `lag 3` = `MAM`

Untuk notebook ini, `lag` dipakai sebagai pergeseran bulan pada seri Nino-3.4 yang sudah dirata-ratakan 3 bulan.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.ndimage import gaussian_filter
from scipy.stats import norm, t as student_t

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
})

PROJECT_ROOT = Path('../../..').resolve()
DATA_ROOT = Path('../../external/ClimateData').resolve()
CACHE_DIR = Path('../../data/intermediate/divided_regression').resolve()
RESULTS_DIR = Path('../../results/analisis_regresi_26-19').resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAINFALL_PATH = Path('../../external/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
NINO34_PATH = Path('../../external/ClimateData/index-monthly/nino34.anom.csv').resolve()
OUT_NC = CACHE_DIR / 'rainfall_spatial_reg_cache_1981_2025.nc'

START_YEAR = 1981
END_YEAR = 2025
P1 = (1981, 2006)
P2 = (2007, 2025)
LAGS = np.arange(-12, 13)
PLOT_LAG = 0

MC_EXTENT = [80, 160, -20, 20]
TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}
MC_XTICKS = np.arange(90, 161, 20)
MC_YTICKS = np.arange(-20, 21, 10)
CORR_LEVELS = np.arange(-1, 1.01, 0.05)
CORR_TICKS = np.arange(-1, 1.01, 0.25)

LAG_TITLES = {
    -12: '-DJF',
    -11: '-JFM',
    -10: '-FMA',
    -9: '-MAM',
    -8: '-AMJ',
    -7: '-MJJ',
    -6: '-JJA',
    -5: '-JAS',
    -4: '-ASO',
    -3: '-SON',
    -2: '-OND',
    -1: '-NDJ',
    0: 'DJF',
    1: 'JFM',
    2: 'FMA',
    3: 'MAM',
    4: 'AMJ',
    5: 'MJJ',
    6: 'JJA',
    7: 'JAS',
    8: 'ASO',
    9: 'SON',
    10: 'OND',
    11: 'NDJ',
    12: 'DJF+',
}


def lag_label(lag: int) -> str:
    return LAG_TITLES.get(int(lag), f'{int(lag):+d}')


def save_plot(fig, filename):
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')


def smooth_for_plot(da, sigma=0.7):
    if not {'lat', 'lon'}.issubset(da.dims):
        return da
    def _smooth(arr):
        return gaussian_filter(arr, sigma=sigma)
    smoothed = xr.apply_ufunc(
        _smooth,
        da,
        input_core_dims=[['lat', 'lon']],
        output_core_dims=[['lat', 'lon']],
        vectorize=True,
        dask='allowed',
        output_dtypes=[da.dtype],
    )
    smoothed = smoothed.transpose(*da.dims)
    smoothed = smoothed.assign_attrs(da.attrs)
    if da.name is not None:
        smoothed = smoothed.rename(da.name)
    return smoothed


def standardize_rainfall_da(da: xr.DataArray) -> xr.DataArray:
    rename_map = {}
    if 'latitude' in da.dims or 'latitude' in da.coords:
        rename_map['latitude'] = 'lat'
    if 'longitude' in da.dims or 'longitude' in da.coords:
        rename_map['longitude'] = 'lon'
    if 'valid_time' in da.dims or 'valid_time' in da.coords:
        rename_map['valid_time'] = 'time'
    if rename_map:
        da = da.rename(rename_map)
    da = da.assign_coords(time=pd.to_datetime(da['time'].values)).sortby('time')
    da = da.assign_coords(lon=(da['lon'] % 360)).sortby('lon')
    return da


def load_nino34_monthly(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise ValueError(f'Nino3.4 file must have at least 2 columns, got {df.columns.tolist()}')
    date_col = None
    value_col = None
    for col in df.columns:
        low = str(col).lower()
        if date_col is None and low in {'date', 'time', 'month', 'ym'}:
            date_col = col
        elif value_col is None and low in {'nino34', 'nino3.4', 'nino-3.4', 'anom', 'anomaly', 'value'}:
            value_col = col
    if date_col is None:
        date_col = df.columns[0]
    if value_col is None:
        value_col = df.columns[1]
    s = pd.Series(pd.to_numeric(df[value_col], errors='coerce').to_numpy(), index=pd.to_datetime(df[date_col]))
    s = s.sort_index().groupby(s.index.to_period('M')).mean()
    s.index = s.index.to_timestamp(how='start')
    s.name = 'nino34'
    return s


def build_centered_3mo_running_mean(series: pd.Series) -> pd.Series:
    return series.rolling(3, center=True, min_periods=3).mean().dropna()


def build_djf_year_coord(time: xr.DataArray) -> xr.DataArray:
    return xr.where(time.dt.month == 12, time.dt.year + 1, time.dt.year)


def prep_rainfall_field(path: Path) -> xr.DataArray:
    ds = xr.open_dataset(path)
    rain_var = 'precipitation' if 'precipitation' in ds.data_vars else list(ds.data_vars)[0]
    rain = standardize_rainfall_da(ds[rain_var])
    rain = rain.sel(time=slice('1979-12-01', '2025-02-28'))
    rain = rain.sel(lat=slice(20, -20), lon=slice(80, 160))
    rain = rain.chunk({'time': 36, 'lat': 120, 'lon': 120}).astype(np.float32)
    rain_djf = rain.sel(time=rain.time.dt.month.isin([12, 1, 2]))
    rain_djf = rain_djf.assign_coords(djf_year=('time', build_djf_year_coord(rain_djf.time).data))
    return rain_djf


def build_nino_lag_series(nino_3mo: pd.Series, years: np.ndarray, lag: int) -> xr.DataArray:
    vals = []
    idx = []
    for year in years:
        ts = pd.Timestamp(int(year), 1, 1) + pd.DateOffset(months=int(lag))
        if ts in nino_3mo.index:
            vals.append(float(nino_3mo.loc[ts]))
            idx.append(int(year))
    return xr.DataArray(np.array(vals, dtype=float), coords={'djf_year': np.array(idx, dtype=int)}, dims='djf_year', name=f'nino34_lag_{lag:+d}')


def build_nino_lag_matrix(nino_3mo: pd.Series, years: np.ndarray, lags: np.ndarray) -> xr.DataArray:
    year_index = np.array(years, dtype=int)
    lag_series = []
    for lag in np.array(lags, dtype=int):
        lag_da = build_nino_lag_series(nino_3mo, year_index, int(lag)).reindex(djf_year=year_index)
        lag_series.append(lag_da.assign_coords(lag=int(lag)).expand_dims('lag'))
    lag_matrix = xr.concat(lag_series, dim='lag').transpose('lag', 'djf_year')
    return lag_matrix.astype(np.float32)


def plot_mc_map(data, sig_mask, title, filename, cbar_label):
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))
    plot_data = smooth_for_plot(data).reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=CORR_LEVELS,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask <= 0.05).fillna(False).astype(np.int8)
    sig_plot = sig_plot.coarsen(lat=8, lon=8, boundary='trim').max() > 0
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.set_extent(MC_EXTENT, crs=ccrs.PlateCarree())
    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(MC_XTICKS, crs=ccrs.PlateCarree())
    ax.set_yticks(MC_YTICKS, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=20)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=CORR_TICKS, spacing='proportional', extend='both')
    cbar.set_label(cbar_label, fontsize=13, labelpad=10)
    cbar.ax.tick_params(labelsize=12)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# bikin dataset lagged regressionnya run 1x saja


In [ ]:
def _finite_mask(*arrays):
    mask = None
    for arr in arrays:
        current = xr.apply_ufunc(np.isfinite, arr)
        mask = current if mask is None else (mask & current)
    return mask


def simple_regression_stats(field, index, sample_dim):
    field, index = xr.align(field, index, join='inner')
    valid = _finite_mask(field, index)
    x = index.where(valid)
    y = field.where(valid)

    n = valid.sum(sample_dim).astype(np.float32)
    safe_n = xr.where(n > 0, n, np.nan)

    sx = x.sum(sample_dim, skipna=True)
    sy = y.sum(sample_dim, skipna=True)
    sxx = (x * x).sum(sample_dim, skipna=True)
    syy = (y * y).sum(sample_dim, skipna=True)
    sxy = (x * y).sum(sample_dim, skipna=True)

    sxx_c = sxx - (sx ** 2 / safe_n)
    sxy_c = sxy - (sx * sy / safe_n)
    syy_c = syy - (sy ** 2 / safe_n)

    slope = sxy_c / sxx_c
    df = safe_n - 2
    sse = syy_c - (sxy_c ** 2 / sxx_c)
    mse = sse / df
    se = np.sqrt(mse / sxx_c)
    t_stat = slope / se
    p = xr.apply_ufunc(
        lambda t, dof: 2 * student_t.sf(np.abs(t), df=dof),
        t_stat,
        df,
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float],
    )

    valid_stats = (n >= 3) & np.isfinite(slope) & np.isfinite(p)
    slope = slope.where(valid_stats)
    p = p.where(valid_stats)
    return slope.astype(np.float32), p.astype(np.float32), n


def pooled_ols_interaction_stats(field_past, field_recent, index_past, index_recent, sample_dim):
    field_past, index_past = xr.align(field_past, index_past, join='inner')
    field_recent, index_recent = xr.align(field_recent, index_recent, join='inner')

    valid_past = _finite_mask(field_past, index_past)
    valid_recent = _finite_mask(field_recent, index_recent)

    y0 = field_past.where(valid_past)
    x0 = index_past.where(valid_past)
    y1 = field_recent.where(valid_recent)
    x1 = index_recent.where(valid_recent)

    def _pooled_ols_1d(y0, x0, y1, x1):
        y0 = np.asarray(y0, dtype=float)
        x0 = np.asarray(x0, dtype=float)
        y1 = np.asarray(y1, dtype=float)
        x1 = np.asarray(x1, dtype=float)

        valid0 = np.isfinite(y0) & np.isfinite(x0)
        valid1 = np.isfinite(y1) & np.isfinite(x1)
        y = np.concatenate([y0[valid0], y1[valid1]])
        x = np.concatenate([x0[valid0], x1[valid1]])
        d = np.concatenate([
            np.zeros(valid0.sum(), dtype=float),
            np.ones(valid1.sum(), dtype=float),
        ])

        n = y.size
        if n < 4:
            return np.nan, np.nan, np.nan

        X = np.column_stack([np.ones(n), x, d, x * d])
        try:
            beta, *_ = np.linalg.lstsq(X, y, rcond=None)
            resid = y - X @ beta
            rss = float(np.sum(resid ** 2))
            df = float(n - X.shape[1])
            if df <= 0:
                return np.nan, np.nan, df
            xtx_inv = np.linalg.inv(X.T @ X)
            sigma2 = rss / df
            se = float(np.sqrt(sigma2 * xtx_inv[3, 3]))
            diff = float(beta[3])
            if not np.isfinite(diff) or not np.isfinite(se) or se == 0:
                return np.nan, np.nan, df
            t_stat = diff / se
            p = float(2 * student_t.sf(np.abs(t_stat), df=df))
            return diff, p, df
        except np.linalg.LinAlgError:
            return np.nan, np.nan, np.nan

    diff, p, df = xr.apply_ufunc(
        _pooled_ols_1d,
        y0,
        x0,
        y1,
        x1,
        input_core_dims=[[sample_dim], [sample_dim], [sample_dim], [sample_dim]],
        output_core_dims=[[], [], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float, float],
    )
    return diff.astype(np.float32), p.astype(np.float32), df.astype(np.float32)
def reg_map_lags(da_djf: xr.DataArray, nino_lag_matrix: xr.DataArray):
    reg_list = []
    p_list = []
    n_list = []
    for lag in nino_lag_matrix['lag'].values:
        reg, pval, n = simple_regression_stats(da_djf, nino_lag_matrix.sel(lag=lag), 'djf_year')
        reg_list.append(reg.expand_dims(lag=[int(lag)]))
        p_list.append(pval.expand_dims(lag=[int(lag)]))
        n_list.append(n.expand_dims(lag=[int(lag)]))
    reg = xr.concat(reg_list, dim='lag').transpose('lag', 'lat', 'lon')
    p = xr.concat(p_list, dim='lag').transpose('lag', 'lat', 'lon')
    n = xr.concat(n_list, dim='lag').transpose('lag', 'lat', 'lon')
    return reg.astype(np.float32), p.astype(np.float32), n.astype(np.float32)


def symmetric_levels(*arrays, n_levels=21, n_ticks=9):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        vals = np.asarray(arr).ravel()
        vals = vals[np.isfinite(vals)]
        if vals.size:
            values.append(vals)
    if values:
        merged = np.concatenate(values)
        absmax = float(np.nanmax(np.abs(merged)))
    else:
        absmax = 1.0
    if not np.isfinite(absmax) or np.isclose(absmax, 0.0):
        absmax = 1.0
    levels = np.linspace(-absmax, absmax, n_levels)
    ticks = np.linspace(-absmax, absmax, n_ticks)
    return levels, ticks, absmax


In [ ]:
# Load data
rain_djf = prep_rainfall_field(RAINFALL_PATH)
nino_monthly = load_nino34_monthly(NINO34_PATH)
nino_3mo = build_centered_3mo_running_mean(nino_monthly)
nino_3mo = nino_3mo.loc['1980-01-01':'2026-01-01']

full_years = np.arange(START_YEAR, END_YEAR + 1)
past_years = np.arange(P1[0], P1[1] + 1)
recent_years = np.arange(P2[0], P2[1] + 1)

rain_djf_annual = rain_djf.groupby('djf_year').mean('time').chunk({'djf_year': -1, 'lat': 120, 'lon': 120})

rain_clim = rain_djf_annual.where(rain_djf_annual.coords['djf_year'].isin(full_years), drop=True)
rain_past = rain_clim.where(rain_clim.coords['djf_year'].isin(past_years), drop=True)
rain_recent = rain_clim.where(rain_clim.coords['djf_year'].isin(recent_years), drop=True)

print('Rainfall path:', RAINFALL_PATH)
print('Nino3.4 path:', NINO34_PATH)
print('Rainfall DJF shape:', rain_clim.shape)


In [ ]:
# Compute lagged spatial regressions and save to cache
nino_lag_full = build_nino_lag_matrix(nino_3mo, full_years, LAGS)
nino_lag_past = nino_lag_full.sel(djf_year=past_years)
nino_lag_recent = nino_lag_full.sel(djf_year=recent_years)

reg_clim, p_clim, n_clim = reg_map_lags(rain_clim, nino_lag_full)
reg_past, p_past, n_past = reg_map_lags(rain_past, nino_lag_past)
reg_recent, p_recent, n_recent = reg_map_lags(rain_recent, nino_lag_recent)

reg_diff_list = []
p_diff_list = []
for lag in LAGS:
    reg_diff, p_diff, _ = pooled_ols_interaction_stats(
        rain_past,
        rain_recent,
        nino_lag_past.sel(lag=lag),
        nino_lag_recent.sel(lag=lag),
        'djf_year',
    )
    reg_diff_list.append(reg_diff.expand_dims(lag=[int(lag)]))
    p_diff_list.append(p_diff.expand_dims(lag=[int(lag)]))

reg_recent_minus_past = xr.concat(reg_diff_list, dim='lag').transpose('lag', 'lat', 'lon').astype(np.float32)
p_recent_minus_past = xr.concat(p_diff_list, dim='lag').transpose('lag', 'lat', 'lon').astype(np.float32)

spatial_reg = xr.Dataset(
    {
        'rain_reg_clim': reg_clim,
        'rain_reg_past': reg_past,
        'rain_reg_recent': reg_recent,
        'rain_reg_recent_minus_past': reg_recent_minus_past,
        'rain_reg_clim_p': p_clim,
        'rain_reg_past_p': p_past,
        'rain_reg_recent_p': p_recent,
        'rain_reg_recent_minus_past_p': p_recent_minus_past,
        'rain_reg_clim_sig': (p_clim < 0.05),
        'rain_reg_past_sig': (p_past < 0.05),
        'rain_reg_recent_sig': (p_recent < 0.05),
        'rain_reg_recent_minus_past_sig': (p_recent_minus_past < 0.05),
        'rain_reg_clim_n': n_clim,
        'rain_reg_past_n': n_past,
        'rain_reg_recent_n': n_recent,
    }
)

encoding = {
    var_name: {'zlib': True, 'complevel': 3, 'dtype': 'float32'}
    for var_name in spatial_reg.data_vars
}
spatial_reg.to_netcdf(OUT_NC, encoding=encoding)
print('Saved cache:', OUT_NC)
print(spatial_reg)


## Plot Lag Terpilih

Notebook ini hanya mem-plot satu lag yang dipilih lewat `PLOT_LAG`. Kalau ingin melihat lag lain, ubah nilainya lalu jalankan ulang sel plot saja.


In [ ]:
spatial_reg = xr.open_dataset(OUT_NC)

REG_LEVELS, REG_TICKS, REG_ABSMAX = symmetric_levels(
    spatial_reg['rain_reg_clim'],
    spatial_reg['rain_reg_past'],
    spatial_reg['rain_reg_recent'],
    spatial_reg['rain_reg_recent_minus_past'],
)
print('Loaded cache:', OUT_NC)
print('Regression color absmax:', REG_ABSMAX)


In [ ]:
# Plot the selected lag using the same map style as plot_regression_mc.ipynb
plot_lag_label = lag_label(PLOT_LAG)
plot_lag_index = int(np.where(LAGS == PLOT_LAG)[0][0])

rain_plot_clim = spatial_reg['rain_reg_clim'].isel(lag=plot_lag_index)
rain_plot_past = spatial_reg['rain_reg_past'].isel(lag=plot_lag_index)
rain_plot_recent = spatial_reg['rain_reg_recent'].isel(lag=plot_lag_index)
rain_plot_diff = spatial_reg['rain_reg_recent_minus_past'].isel(lag=plot_lag_index)

rain_plot_clim_sig = spatial_reg['rain_reg_clim_sig'].isel(lag=plot_lag_index)
rain_plot_past_sig = spatial_reg['rain_reg_past_sig'].isel(lag=plot_lag_index)
rain_plot_recent_sig = spatial_reg['rain_reg_recent_sig'].isel(lag=plot_lag_index)
rain_plot_diff_sig = spatial_reg['rain_reg_recent_minus_past_sig'].isel(lag=plot_lag_index)


In [ ]:
# Load cache
spatial_reg = xr.open_dataset(OUT_NC)

# Select 5 lags: -JJA, -SON, DJF, +MAM, +JJA
plot_lags = [-6, -3, 0, 3, 6]
lag_labels_list = ['-JJA', '-SON', 'DJF', '+MAM', '+JJA']

# Create 3x5 grid (3 rows: P1, P2, Diff; 5 columns: lags)
fig, axes = plt.subplots(3, 5, figsize=(20, 12), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
fig.subplots_adjust(left=0.12, right=0.98, bottom=0.14, top=0.92, wspace=0.04, hspace=0.08)

row_specs = [
    ('rain_reg_past', 'rain_reg_past_sig', f'P1 ({P1[0]}-{P1[1]})'),
    ('rain_reg_recent', 'rain_reg_recent_sig', f'P2 ({P2[0]}-{P2[1]})'),
    ('rain_reg_recent_minus_past', 'rain_reg_recent_minus_past_sig', 'P2 - P1'),
]

# Plot loop: 3 rows x 5 columns
for row, (var_name, var_sig_name, row_title) in enumerate(row_specs):
    for col, (lag, lag_label) in enumerate(zip(plot_lags, lag_labels_list)):
        ax = axes[row, col]

        data = spatial_reg[var_name].sel(lag=lag).reset_coords(drop=True)
        sig = spatial_reg[var_sig_name].sel(lag=lag)

        plot_data = smooth_for_plot(data)
        if hasattr(plot_data, 'compute'):
            plot_data = plot_data.compute()

        plot_data.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap='bwr',
            levels=REG_LEVELS,
            extend='both',
            add_colorbar=False,
            infer_intervals=True,
            add_labels=False,
        )

        sig_plot = sig.fillna(False).astype(np.int8)
        sig_plot = sig_plot.coarsen(lat=20, lon=20, boundary='trim').max() > 0
        yy, xx = np.where(sig_plot.values)
        if yy.size > 0:
            ax.scatter(
                sig_plot['lon'].values[xx],
                sig_plot['lat'].values[yy],
                s=15,
                c='k',
                marker='.',
                linewidths=0,
                alpha=0.7,
                transform=ccrs.PlateCarree(),
                zorder=4,
            )

        ax.set_extent(MC_EXTENT, crs=ccrs.PlateCarree())
        ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=3)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.3, color='black', zorder=3)
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
        ax.set_xticks(MC_XTICKS, crs=ccrs.PlateCarree())
        ax.set_yticks(MC_YTICKS, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(direction='out', top=False, right=False, labelsize=16, labelbottom=(row == 2), labelleft=(col == 0))

        if row == 0:
            ax.set_title(lag_label, fontsize=20, fontweight='bold', loc='center', pad=6)

        ax.set_xlabel('')
        ax.set_ylabel('')

        if col == 0:
            ax.text(-0.28, 0.5, row_title, transform=ax.transAxes, rotation=90, va='center', ha='center', fontsize=16, fontweight='bold', clip_on=False)



# Shared normalized colorbar using the same levels/ticks
cbar_ax = fig.add_axes([0.18, 0.08, 0.64, 0.025])
cbar = fig.colorbar(
    plt.cm.ScalarMappable(cmap='bwr', norm=plt.Normalize(vmin=REG_LEVELS[0], vmax=REG_LEVELS[-1])),
    cax=cbar_ax,
    orientation='horizontal',
    ticks=REG_TICKS,
    boundaries=REG_LEVELS,
    spacing='proportional',
    extend='both',
)
cbar.set_label('Slope regresi rainfall', fontsize=16, labelpad=10)
cbar.ax.tick_params(labelsize=16)

fig.suptitle('Spatial Regression: Rainfall DJF vs Nino3.4 (5 Lags × 3 Periods)', fontsize=16, fontweight='bold', y=0.98)
fig.savefig(RESULTS_DIR / 'rainfall_spatial_reg_5x3_alllags.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: rainfall_spatial_reg_5x3_alllags.png')


In [ ]:
# Load cache for target-box plot
spatial_reg = xr.open_dataset(OUT_NC)

plot_lags = [-6, -3, 0, 3, 6]
lag_labels_list = ['-JJA', '-SON', 'DJF', '+MAM', '+JJA']
target_extent = [TARGET_BOX['lon_min'], TARGET_BOX['lon_max'], TARGET_BOX['lat_min'], TARGET_BOX['lat_max']]
target_xticks = np.arange(100, 121, 10)
target_yticks = np.arange(-6, 3, 2)

fig, axes = plt.subplots(3, 5, figsize=(20, 12), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
fig.subplots_adjust(left=0.12, right=0.98, bottom=0.14, top=0.92, wspace=0.04, hspace=0.08)

row_specs = [
    ('rain_reg_past', 'rain_reg_past_sig', f'P1 ({P1[0]}-{P1[1]})'),
    ('rain_reg_recent', 'rain_reg_recent_sig', f'P2 ({P2[0]}-{P2[1]})'),
    ('rain_reg_recent_minus_past', 'rain_reg_recent_minus_past_sig', 'P2 - P1'),
]

for row, (var_name, var_sig_name, row_title) in enumerate(row_specs):
    for col, (lag, lag_label) in enumerate(zip(plot_lags, lag_labels_list)):
        ax = axes[row, col]
        data = spatial_reg[var_name].sel(lag=lag).reset_coords(drop=True).sel(
            lon=slice(TARGET_BOX['lon_min'], TARGET_BOX['lon_max']),
            lat=slice(TARGET_BOX['lat_max'], TARGET_BOX['lat_min']),
        )
        sig = spatial_reg[var_sig_name].sel(lag=lag).sel(
            lon=slice(TARGET_BOX['lon_min'], TARGET_BOX['lon_max']),
            lat=slice(TARGET_BOX['lat_max'], TARGET_BOX['lat_min']),
        )

        plot_data = smooth_for_plot(data)
        if hasattr(plot_data, 'compute'):
            plot_data = plot_data.compute()

        plot_data.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap='bwr',
            levels=REG_LEVELS,
            extend='both',
            add_colorbar=False,
            infer_intervals=True,
            add_labels=False,
        )

        sig_plot = sig.fillna(False).astype(np.int8)
        sig_plot = sig_plot.coarsen(lat=8, lon=8, boundary='trim').max() > 0
        yy, xx = np.where(sig_plot.values)
        if yy.size > 0:
            ax.scatter(
                sig_plot['lon'].values[xx],
                sig_plot['lat'].values[yy],
                s=12,
                c='k',
                marker='.',
                linewidths=0,
                alpha=0.7,
                transform=ccrs.PlateCarree(),
                zorder=4,
            )

        ax.set_extent(target_extent, crs=ccrs.PlateCarree())
        ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=3)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.3, color='black', zorder=3)
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
        ax.set_xticks(target_xticks, crs=ccrs.PlateCarree())
        ax.set_yticks(target_yticks, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(direction='out', top=False, right=False, labelsize=16, labelbottom=(row == 2), labelleft=(col == 0))

        if row == 0:
            ax.set_title(lag_label, fontsize=20, fontweight='bold', loc='center', pad=6)
        if col == 0:
            ax.text(-0.28, 0.5, row_title, transform=ax.transAxes, rotation=90, va='center', ha='center', fontsize=16, fontweight='bold', clip_on=False)

        ax.set_xlabel('')
        ax.set_ylabel('')

cbar_ax = fig.add_axes([0.18, 0.08, 0.64, 0.025])
cbar = fig.colorbar(
    plt.cm.ScalarMappable(cmap='bwr', norm=plt.Normalize(vmin=REG_LEVELS[0], vmax=REG_LEVELS[-1])),
    cax=cbar_ax,
    orientation='horizontal',
    ticks=REG_TICKS,
    boundaries=REG_LEVELS,
    spacing='proportional',
    extend='both',
)
cbar.set_label('Slope regresi rainfall', fontsize=16, labelpad=10)
cbar.ax.tick_params(labelsize=16)

fig.suptitle('Spatial Regression: Rainfall DJF vs Nino3.4 (Target Box, 5 Lags × 3 Periods)', fontsize=16, fontweight='bold', y=0.98)
fig.savefig(RESULTS_DIR / 'rainfall_spatial_reg_targetbox_5x3_alllags.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: rainfall_spatial_reg_targetbox_5x3_alllags.png')
